# Testing Texas Hold'em deals

This notebook tests private hands, the community board, burn cards, and validation.

In [ ]:
import random
import sys
from pathlib import Path

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "cards.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from cards import Deck, hand_names, make_hand
from poker import HoldemDeal, deal_holdem, describe_deal

print("PASS: imports successful")

## Deal a six-player game

In [ ]:
deck = Deck(rng=random.Random(42))
deal = deal_holdem(6, deck=deck)

print(describe_deal(deal))
print("Burn cards:", " ".join(hand_names(deal.burn_cards)))
print("Cards left in deck:", len(deck))

In [ ]:
assert len(deal.private_hands) == 6
assert all(hand.bit_count() == 2 for hand in deal.private_hands)
assert deal.flop.bit_count() == 3
assert deal.turn.bit_count() == 1
assert deal.river.bit_count() == 1
assert deal.board.bit_count() == 5
assert deal.burn_cards.bit_count() == 3
assert deal.all_dealt_cards.bit_count() == 20
assert len(deck) == 32
print("PASS: deal sizes are correct")

## Check that no cards overlap

In [ ]:
groups = [*deal.private_hands, deal.flop, deal.turn, deal.river, deal.burn_cards]
for index, first in enumerate(groups):
    for second in groups[index + 1:]:
        assert first & second == 0

assert deal.player_hand(0).bit_count() == 2
assert deal.player_cards(0).bit_count() == 7
print("PASS: every card is unique and players have seven available cards")

## Deal without burn cards

In [ ]:
unburned_deck = Deck(rng=random.Random(7))
unburned_deal = deal_holdem(2, deck=unburned_deck, burn=False)
assert unburned_deal.burn_cards == 0
assert unburned_deal.all_dealt_cards.bit_count() == 9
assert len(unburned_deck) == 43
print("PASS: dealing without burn cards")

## Validation tests

In [ ]:
def assert_raises(error_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except error_type:
        return
    raise AssertionError(f"{function.__name__} did not raise {error_type.__name__}")

assert_raises(ValueError, deal_holdem, 1)
assert_raises(ValueError, deal_holdem, 11)
assert_raises(TypeError, deal_holdem, 2.5)

duplicate = make_hand(["As", "Kd"])
assert_raises(
    ValueError, HoldemDeal, (duplicate, duplicate),
    make_hand(["2c", "3c", "4c"]), make_hand(["5c"]), make_hand(["6c"]),
)
print("PASS: invalid deals are rejected")